In [1]:
# Run this cell once after restarting the kernel
%pip install --upgrade pip
%pip install "numpy<2" ultralytics

     ---------------------------------------- 1.8/1.8 MB 2.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 22.3.1
    Uninstalling pip-22.3.1:
      Successfully uninstalled pip-22.3.1
Note: you may need to restart the kernel to use updated packages.

INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
  Using cached opencv_python-4.13.0.90-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (19 kB)
  Using cached opencv_python-4.11.0.86-cp37-abi3-win_amd64.whl.metadata (20 kB)
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ----------------- ---------------------- 0.5/1.2 MB 2.8 MB/s eta 0:00:01
   ---------------------------------- ----- 1.0/1.2 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 2.6 MB/s  0:00:00
Using cached opencv_python-4

  You can safely remove it manually.


In [12]:
# NVIDIA Quadro T1000 (CUDA) setup for Ultralytics
# 1) Run this cell
# 2) Restart kernel (Kernel > Restart)
# 3) Run the CUDA status cell (next cell)
%pip uninstall -y torch torchvision torchaudio
%pip install --upgrade pip
%pip install --index-url https://download.pytorch.org/whl/cu121 torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2
# If cu121 fails due to driver constraints, try cu118 instead:
# %pip install --index-url https://download.pytorch.org/whl/cu118 torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2

Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu121Note: you may need to restart the kernel to use updated packages.

     ---------------------------------------- 0.0/2.5 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.5 GB 2.4 MB/s eta 0:17:01
     ---------------------------------------- 0.0/2.5 GB 2.3 MB/s eta 0:17:54
     ---------------------------------------- 0.0/2.5 GB 2.8 MB/s eta 0:14:39
     -------------------------

In [ ]:
import os
import subprocess
import torch

# If this env var is set to an empty string, it can hide all GPUs.
# Remove it (then restart kernel) if you want torch to see your GPU(s).
print('CUDA_VISIBLE_DEVICES repr:', repr(os.environ.get('CUDA_VISIBLE_DEVICES')))

print('torch:', torch.__version__)
print('torch.version.cuda:', torch.version.cuda)
print('torch.cuda.is_available():', torch.cuda.is_available())
print('torch.cuda.device_count():', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU 0:', torch.cuda.get_device_name(0))

try:
    out = subprocess.check_output(['nvidia-smi'], stderr=subprocess.STDOUT, text=True)
    print('\n--- nvidia-smi ---')
    print(out.splitlines()[0])
except Exception as e:
    print('\nnvidia-smi not available:', e)

torch: 2.0.1+cpu
torch.version.cuda: None
torch.cuda.is_available(): False
torch.cuda.device_count(): 0
CUDA_VISIBLE_DEVICES: 0

--- nvidia-smi ---
Fri Apr 10 04:35:28 2026       


In [9]:
import numpy as np
import ultralytics
import torch

print('numpy:', np.__version__)
print('ultralytics:', ultralytics.__version__)
print('torch:', torch.__version__)
print('torch.version.cuda:', torch.version.cuda)
print('torch.cuda.is_available():', torch.cuda.is_available())

numpy: 1.26.4
ultralytics: 8.4.36
torch: 2.0.1+cpu
torch.version.cuda: None
torch.cuda.is_available(): False


In [45]:
import os
import random
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter,ImageEnhance
import arabic_reshaper
from bidi.algorithm import get_display

# --- 1. STRICT CLASSES (Using correct 'هـ') ---
CLASSES = {
    '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9,
    'أ': 10, 'ب': 11, 'د': 12, 'هـ': 13, 'و': 14, 'ط': 15, 'ي': 16,
    'W': 17 
}

IMG_DIR = "dataset/images"
LBL_DIR = "dataset/labels"
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(LBL_DIR, exist_ok=True)

try:
    FONT_NUMBERS = ImageFont.truetype("arial.ttf", 65)
    FONT_ARABIC = ImageFont.truetype("arial.ttf", 60)
except IOError:
    print("ERROR: Please place an 'arial.ttf' font file in the directory.")
    exit()

def get_yolo_format(bbox, img_width, img_height):
    x_min, y_min, x_max, y_max = bbox
    x_center = (x_min + x_max) / 2.0 / img_width
    y_center = (y_min + y_max) / 2.0 / img_height
    w = (x_max - x_min) / img_width
    h = (y_max - y_min) / img_height
    return x_center, y_center, w, h

def draw_text_and_get_bbox(draw, text, position, font, img_width, img_height, class_id, yolo_labels, text_color="black"):
    bbox = draw.textbbox(position, text, font=font)
    draw.text(position, text, fill=text_color, font=font)
    x_c, y_c, w, h = get_yolo_format(bbox, img_width, img_height)
    yolo_labels.append(f"{class_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}")

def draw_centered_text(draw, text_string, block_x_min, block_x_max, y_pos, font, img_width, img_height, yolo_labels, text_color="black", is_arabic=False, arabic_class_id=None):
    total_width = draw.textlength(text_string, font=font)
    block_width = block_x_max - block_x_min
    current_x = block_x_min + (block_width - total_width) / 2.0

    if is_arabic:
        draw_text_and_get_bbox(draw, text_string, (current_x, y_pos), font, img_width, img_height, arabic_class_id, yolo_labels, text_color)
    else:
        for char in text_string:
            # Skip spaces for bounding boxes (e.g., in "W 18")
            if char == " ":
                current_x += 15
                continue
            draw_text_and_get_bbox(draw, char, (current_x, y_pos), font, img_width, img_height, CLASSES[char], yolo_labels, text_color)
            char_width = draw.textlength(char, font=font)
            current_x += char_width + 4 # Natural kerning space

def generate_strict_plate(image_id, plate_type="standard"):
    yolo_labels = []
    
    # --- PHYSICAL DIMENSIONS ---
    if plate_type in ["standard", "ww", "w18"]:
        WIDTH, HEIGHT = 520, 110
    elif plate_type == "motorcycle":
        WIDTH, HEIGHT = 360, 200 # Square ratio based on real images

    img = Image.new('RGB', (WIDTH, HEIGHT), color='white')
    draw = ImageDraw.Draw(img)

    # --- RENDER LOGIC BY TYPE ---
    if plate_type == "standard":
        # Two vertical separator lines
        draw.line([(340, 15), (340, 95)], fill="black", width=4)
        draw.line([(420, 15), (420, 95)], fill="black", width=4)
        
        # Standard Black Border
        draw.rectangle([2, 2, WIDTH-2, HEIGHT-2], outline="black", width=4)

        # Left Block: Registration Number
        reg_num = str(random.randint(1, 99999))
        draw_centered_text(draw, reg_num, 0, 340, 15, FONT_NUMBERS, WIDTH, HEIGHT, yolo_labels)

        # Middle Block: Arabic Letter
        plate_letter = random.choice(['أ', 'ب', 'د', 'هـ', 'و', 'ط', 'ي'])
        bidi_text = get_display(arabic_reshaper.reshape(plate_letter))
        draw_centered_text(draw, bidi_text, 340, 420, 10, FONT_ARABIC, WIDTH, HEIGHT, yolo_labels, is_arabic=True, arabic_class_id=CLASSES[plate_letter])

        # Right Block: City Code
        city_code = str(random.randint(1, 89))
        draw_centered_text(draw, city_code, 420, 520, 15, FONT_NUMBERS, WIDTH, HEIGHT, yolo_labels)

    elif plate_type == "motorcycle":
        # Horizontal and Vertical separators
        draw.line([(10, 100), (350, 100)], fill="black", width=4) # Horizontal
        draw.line([(180, 10), (180, 100)], fill="black", width=4) # Vertical (top half only)
        
        # Standard Black Border
        draw.rectangle([2, 2, WIDTH-2, HEIGHT-2], outline="black", width=4)

        # Top Left: Arabic Letter
        plate_letter = random.choice(['أ', 'ب', 'د', 'هـ', 'و', 'ط', 'ي'])
        bidi_text = get_display(arabic_reshaper.reshape(plate_letter))
        draw_centered_text(draw, bidi_text, 0, 180, 15, FONT_ARABIC, WIDTH, HEIGHT, yolo_labels, is_arabic=True, arabic_class_id=CLASSES[plate_letter])
        
        # Top Right: City Code
        city_code = str(random.randint(1, 89))
        draw_centered_text(draw, city_code, 180, 360, 15, FONT_NUMBERS, WIDTH, HEIGHT, yolo_labels)
            
        # Bottom: Registration Number
        reg_num = str(random.randint(1, 99999))
        draw_centered_text(draw, reg_num, 0, 360, 110, FONT_NUMBERS, WIDTH, HEIGHT, yolo_labels)

    elif plate_type == "ww":
        # NO lines. Black text. Black border.
        draw.rectangle([2, 2, WIDTH-2, HEIGHT-2], outline="black", width=4)
        
        reg_num = str(random.randint(100000, 999999))
        draw_centered_text(draw, reg_num, 0, 350, 15, FONT_NUMBERS, WIDTH, HEIGHT, yolo_labels)
        draw_centered_text(draw, "WW", 350, 520, 15, FONT_NUMBERS, WIDTH, HEIGHT, yolo_labels)

    elif plate_type == "w18":
        # NO lines. Red text. Thick Red border.
        draw.rectangle([3, 3, WIDTH-3, HEIGHT-3], outline="#D32F2F", width=6)
        
        reg_num = str(random.randint(10000, 99999))
        draw_centered_text(draw, reg_num, 0, 300, 15, FONT_NUMBERS, WIDTH, HEIGHT, yolo_labels, text_color="#D32F2F")
        draw_centered_text(draw, "W 18", 300, 520, 15, FONT_NUMBERS, WIDTH, HEIGHT, yolo_labels, text_color="#D32F2F")

# ... (Keep all the previous v5.0 generation logic) ...

    # ==========================================
    # --- ADVANCED ADVERSARIAL AUGMENTATION ---
    # ==========================================
    
    # 1. THE "SWAP" (10% chance to invert colors completely)
    if random.random() < 0.10:
        import PIL.ImageOps
        img = PIL.ImageOps.invert(img)

    # 2. THE "DESERT STORM" (Mud and Screws)
    # Add 2 to 5 random black/brown "mud" splatters or screws
    if random.random() < 0.30: # 30% chance of being dirty
        dirt_draw = ImageDraw.Draw(img)
        for _ in range(random.randint(2, 5)):
            # Random position
            dx = random.randint(0, WIDTH)
            dy = random.randint(0, HEIGHT)
            # Random size of mud splatter
            r = random.randint(2, 8) 
            dirt_draw.ellipse([dx-r, dy-r, dx+r, dy+r], fill=(50, 50, 50))

    # 3. THE "BROKEN HEADLIGHT" (Harsh Contrast/Fading)
    # Sometimes plates are washed out by the sun or incredibly dark
    if random.random() < 0.20:
        enhancer = ImageEnhance.Contrast(img)
        # Randomly make it extremely low contrast (washed out) or high contrast
        img = enhancer.enhance(random.choice([0.4, 0.6, 1.5]))
        
        brightness = ImageEnhance.Brightness(img)
        img = brightness.enhance(random.choice([0.5, 1.2])) # Darken or Overexpose

    # 4. BLUR DYNAMICS (Motion vs. Camera Focus)
    blur_type = random.choice(["none", "gaussian", "motion"])
    
    if blur_type == "gaussian":
        # Out of focus camera
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))
    
    elif blur_type == "motion":
        # Fast moving car (Horizontal directional blur hack using BoxBlur)
        # We stretch the image horizontally, blur it, and shrink it back
        img = img.resize((WIDTH // 2, HEIGHT)) # squish
        img = img.filter(ImageFilter.BoxBlur(1)) # blur
        img = img.resize((WIDTH, HEIGHT)) # stretch back (creates horizontal motion effect)

    # --- EXPORT ---
    img_path = os.path.join(IMG_DIR, f"plate_{image_id}_{plate_type}.jpg")
    lbl_path = os.path.join(LBL_DIR, f"plate_{image_id}_{plate_type}.txt")
    
    img.save(img_path)
    with open(lbl_path, "w", encoding="utf-8") as f:
        f.write("\n".join(yolo_labels))

# --- EXECUTION ---
if __name__ == "__main__":
    print("Generating the ultimate Moroccan license plate dataset...")
    
    # Generate a robust mix of 200 plates to test
    for i in range(5000):
        generate_strict_plate(i, plate_type="standard")
        generate_strict_plate(i+5000, plate_type="motorcycle")
        generate_strict_plate(i+10000, plate_type="ww")
        generate_strict_plate(i+15000, plate_type="w18")
        
    print("Generation complete! Check your 'dataset/images' and 'dataset/labels' folders.")

Generating the ultimate Moroccan license plate dataset...
Generation complete! Check your 'dataset/images' and 'dataset/labels' folders.


In [2]:
import os
import random
import shutil

def split_yolo_dataset(base_dir="dataset", train_ratio=0.8):
    """
    Splits a flat folder of YOLO images and labels into train/val structure.
    """
    img_dir = os.path.join(base_dir, "images")
    lbl_dir = os.path.join(base_dir, "labels")

    # 1. Create the YOLO directory structure
    for split in ['train', 'val']:
        os.makedirs(os.path.join(img_dir, split), exist_ok=True)
        os.makedirs(os.path.join(lbl_dir, split), exist_ok=True)

    # 2. Get all image files (ignoring directories)
    all_images = [f for f in os.listdir(img_dir) if f.endswith('.jpg') and os.path.isfile(os.path.join(img_dir, f))]

    if len(all_images) == 0:
        print("No .jpg images found in the dataset/images folder!")
        return

    # 3. Shuffle the data randomly
    random.seed(42) # Keeps the randomness consistent if you run it again
    random.shuffle(all_images)

    # 4. Calculate the split index
    split_index = int(len(all_images) * train_ratio)
    train_images = all_images[:split_index]
    val_images = all_images[split_index:]

    # 5. Helper function to move the pairs
    def move_files(file_list, split_name):
        for img_name in file_list:
            lbl_name = img_name.replace('.jpg', '.txt')

            src_img = os.path.join(img_dir, img_name)
            src_lbl = os.path.join(lbl_dir, lbl_name)

            dst_img = os.path.join(img_dir, split_name, img_name)
            dst_lbl = os.path.join(lbl_dir, split_name, lbl_name)

            # Ensure both image and label exist before moving
            if os.path.exists(src_img) and os.path.exists(src_lbl):
                shutil.move(src_img, dst_img)
                shutil.move(src_lbl, dst_lbl)
            else:
                print(f"Warning: Label missing for {img_name}. Skipping.")

    print(f"Found {len(all_images)} total plates.")
    print(f"Splitting: {len(train_images)} for Training, {len(val_images)} for Validation...")

    # 6. Execute the move
    move_files(train_images, 'train')
    move_files(val_images, 'val')

    print("✅ Split complete! Your dataset is now ready for YOLO.")

# --- Execute ---
if __name__ == "__main__":
    split_yolo_dataset()

No .jpg images found in the dataset/images folder!


In [ ]:
import os
import torch
from ultralytics import YOLO

# If CUDA_VISIBLE_DEVICES is set to an empty string, it hides all GPUs.
# Clear it before checking torch.cuda.* (then restart kernel after changing env vars).
if os.environ.get("CUDA_VISIBLE_DEVICES", None) == "":
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)

model = YOLO("yolo11s.pt")

# Uses GPU if CUDA is available, otherwise CPU
device = 0 if torch.cuda.is_available() else "cpu"
print('Selected device:', device)

results = model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    augment=True,
 )

Ultralytics 8.4.36  Python-3.10.9 torch-2.0.1+cpu CPU (Intel Core i7-9850H 2.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, po

KeyboardInterrupt: 

In [11]:
import os, subprocess
import torch

print('torch:', torch.__version__)
print('torch.version.cuda:', torch.version.cuda)
print('torch.cuda.is_available():', torch.cuda.is_available())
print('torch.cuda.device_count():', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))

try:
    out = subprocess.check_output(['nvidia-smi'], stderr=subprocess.STDOUT, text=True)
    print('nvidia-smi:', out.splitlines()[0])
except Exception as e:
    print('nvidia-smi: not available ->', e)

torch: 2.0.1+cpu
torch.version.cuda: None
torch.cuda.is_available(): False
torch.cuda.device_count(): 0
CUDA_VISIBLE_DEVICES: 
nvidia-smi: Fri Apr 10 12:58:53 2026       
